<a href="https://colab.research.google.com/github/ngtan369/Hybrid-Image-Classification/blob/main/notebooks/ex3_imageData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print("Hello world Machine Learning Course !!")

Hello world Machine Learning Course !!


## Download Source Code
 ### github repo: https://github.com/ngtan369/Hybrid-Image-Classification

In [2]:
!rm -rf /content/repo
!git clone https://github.com/ngtan369/Hybrid-Image-Classification /content/repo
%cd /content/repo
import sys
sys.path.insert(0, "/content/repo")
print("Added /content/repo to PYTHONPATH")

Cloning into '/content/repo'...
remote: Enumerating objects: 103, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 103 (delta 30), reused 70 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (103/103), 35.56 KiB | 1.69 MiB/s, done.
Resolving deltas: 100% (30/30), done.
/content/repo
Added /content/repo to PYTHONPATH


# Dowload data
## this part git data set from Kaggle, author: jcoral02, data set name: inriaperson

In [3]:
# Cell 1: Tải dataset từ Kaggle (kagglehub)
# Cài đặt và tải dataset INRIA (chạy trên Colab
%pip install kagglehub -q
import kagglehub, os
print("Downloading INRIA Person from Kaggle...")
dataset_path = kagglehub.dataset_download("jcoral02/inriaperson")
print("Done. dataset_path =", dataset_path)

100%|██████████| 582M/582M [00:04<00:00, 132MB/s]

Extracting files...


Done. dataset_path = /root/.cache/kagglehub/datasets/jcoral02/inriaperson/versions/1


# Traditional Machine Learning Pipeline *italicized text*

Machine Learning Traditional - Full Pipeline (EDA -> Train -> Eval)

## Import Library necessary

In [ ]:
import os, sys, zipfile, shutil
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from IPython.display import Markdown, display
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# --- Configuration for Flexibility ---
CONFIG = {
    "image_size": (224, 224),
    "batch_size": 32,
    "pretrained_model": "resnet50", # Options: vgg16, resnet50, efficientnet
    "classifier": "svm",           # Options: svm, logistic_regression, random_forest
    "repo_path": "/content/repo"
}

# Import custom modules
repo_root = Path(CONFIG["repo_path"]).resolve()
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
from modules.image_utils import load_and_preprocess_data, extract_features_pretrained, save_features_to_disk

print(f"Pipeline configured with {CONFIG['pretrained_model']} and {CONFIG['classifier']}")

## Setup & Data Loading

In [ ]:
if isinstance(dataset_path, str) and dataset_path.endswith('.zip'):
    extract_dir = '/content/inriaperson'
    shutil.rmtree(extract_dir, ignore_errors=True)
    with zipfile.ZipFile(dataset_path,'r') as z: z.extractall(extract_dir)
    dataset_root = extract_dir
else:
    dataset_root = dataset_path

def find_image_root(root):
    p = Path(root)
    for child in p.iterdir():
        if child.is_dir() and any(child.glob('**/*.jpg')): return str(p)
    for sub in p.rglob('*'):
        if sub.is_dir() and any(sub.glob('*.jpg')): return str(sub)
    return str(p)

dataset_for_loader = find_image_root(dataset_root)

## Preprocessing & EDA

In [ ]:
image_size=(224,224); batch_size=32
X_raw, y_raw, class_names = load_and_preprocess_data(dataset_for_loader, image_size=image_size, batch_size=batch_size)

print(f"Loaded {len(X_raw)} images across {len(class_names)} classes: {class_names}")


## Feature Extraction (Transfer Learning features)

In [ ]:
X_features = extract_features_pretrained(X_raw, model_name='resnet50', batch_size=batch_size, pooling='avg')
print("Features extracted shape:", X_features.shape)

## Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_features, y_raw, test_size=0.2, random_state=42, stratify=y_raw)


## Model Training (SVM)

In [ ]:
print("Training SVM model...")
clf = SVC(kernel='linear', C=1.0, probability=True)
clf.fit(X_train, y_train)

## Evaluation


In [ ]:
y_pred = clf.predict(X_test)
display(Markdown("### Kết quả huấn luyện Traditional ML (SVM + ResNet Features)"))
print(classification_report(y_test, y_pred, target_names=class_names))
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

 ## Save results


In [ ]:
prefix = "inria_resnet50_svm"
save_features_to_disk(X_features, y_raw, prefix)
print("Pipeline complete. Features and labels saved with prefix:", prefix)

# Deep Learning

In [ ]:
# Cell 3: Deep Learning - Transfer learning end-to-end (train/val/test)
import tensorflow as tf
from pathlib import Path

# Tạo tf.data từ thư mục (dùng dataset_for_loader)
img_size = image_size
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_for_loader, labels='inferred', label_mode='int',
    image_size=img_size, batch_size=32, validation_split=0.2, subset='training', seed=42)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_for_loader, labels='inferred', label_mode='int',
    image_size=img_size, batch_size=32, validation_split=0.2, subset='validation', seed=42)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

# Build model (transfer learning)
base = tf.keras.applications.ResNet50(weights='imagenet', include_top=False, input_shape=(*img_size,3))
base.trainable = False
inputs = tf.keras.Input(shape=(*img_size,3))
x = tf.keras.applications.resnet.preprocess_input(inputs)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(train_ds.class_names), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print(model.summary())

# Train head
history = model.fit(train_ds, validation_data=val_ds, epochs=5)

# (Optional) Fine-tune some base layers
base.trainable = True
for layer in base.layers[:-20]: layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_ft = model.fit(train_ds, validation_data=val_ds, epochs=5)

# Evaluate and save model
model.evaluate(val_ds)
model.save('models/inria_transfer_resnet50')
print("Saved transfer model to models/inria_transfer_resnet50")